In [1]:
import pandas as pd
import numpy as np
import random
import copy
import math
import warnings

warnings.filterwarnings('ignore')

In [36]:
# cr_df=pd.read_excel('project_CR_Santa Clarita.xlsx',sheet_name='WkDAY-Overall')
cr_df=pd.read_csv('SantaClarita WkDAY-Overall.csv')
wkend_df=pd.read_csv('SantaClarita WkEnd-Overall.csv')
df=pd.read_csv("elvissantaclarita2023obweekday_export_odbc(v1).csv")
ke_df=pd.read_excel("SantaClarita_KINGElvis.xlsx",sheet_name='Elvis_Review')

In [3]:
ke_df=ke_df[ke_df['INTERV_INIT']!='999']
ke_df=ke_df[ke_df['INTERV_INIT']!=999]
ke_df=ke_df[ke_df['1st Cleaner']!='Test/No 5 MIN']
ke_df=ke_df[ke_df['Final_Usage'].str.lower()=='use']

# Getting Data from Database where the Final Usage is Use in KINGELVIS  
df=pd.merge(df,ke_df['id'],on='id',how='inner')
df.drop_duplicates(subset='id',inplace=True)


In [37]:
wkend_df

,LS_NAME,AM Peak,Midday,PM Peak,Evening,Total,Weekend Goals,LS_NAME_CODE
0,1 - Outbound - To Castaic,0,0,0,0,0,6.400,SAN_1_1_01
1,1 - Inbound - To McBean,0,0,0,0,0,0.000,SAN_1_1_00
2,2 - Outbound - To Val Verde,0,0,0,0,0,4.475,SAN_1_2_01
3,2 - Inbound - To McBean,0,0,0,0,0,0.000,SAN_1_2_00
4,3 - Outbound - To Seco Canyon,0,0,0,0,0,5.100,SAN_1_3_01
5,3 - Inbound - To Six Flags Magic Mountain,0,0,0,0,0,0.000,SAN_1_3_00
6,4 - Outbound - To Newhall,0,0,0,0,0,11.600,SAN_1_4_01
7,4 - Inbound - To Bouquet Canyon/Plum Canyon,0,0,0,0,0,0.000,SAN_1_4_00
8,5 - Outbound - To Vasquez Canyon,0,0,0,0,0,21.850,SAN_1_5_01
9,5 - Inbound - To Stevenson Ranch,0,0,0,0,0,0.000,SAN_1_5_00


In [4]:
def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns

In [5]:
#to get the TIMEON column
time_column_check=['timeoncode']
time_column=check_all_characters_present(df,time_column_check)

# to get ROUTE_SURVEYEDCode column from database File
route_survey_column_check=['routesurveyedcode']
route_survey_column=check_all_characters_present(df,route_survey_column_check)

In [6]:
time_column,route_survey_column

(['TIME_ONCode'], ['ROUTE_SURVEYEDCode'])

In [7]:
#values to compare AM, MIDDAY, PM and Evening values
am_values=['AM1','AM2','AM3','AM4']
midday_values=['MID1','MID2','MID3','MID4','MID5']
pm_values=['MID6','PM1','PM2','PM3']
evening_values=['PM4','PM5','PM6','PM7']

In [8]:

cr_df.dropna(subset=['LS_NAME_CODE'],inplace=True)


In [40]:
am_column_check=['ampeak']
pm_column_check=['pmpeak']
midday_column_check=['midday']
evening_column_check=['evening']
overall_goal_column_check=['totalsurveys']
wkend_goal_column_check=['weekendgoals']
am_column=check_all_characters_present(cr_df,am_column_check)
pm_column=check_all_characters_present(cr_df,pm_column_check)
midday_colum=check_all_characters_present(cr_df,midday_column_check)
evening_column=check_all_characters_present(cr_df,evening_column_check)
overall_goal_column=check_all_characters_present(cr_df,overall_goal_column_check)
wkend_am_column=check_all_characters_present(wkend_df,am_column_check)
wkend_pm_column=check_all_characters_present(wkend_df,pm_column_check)
wkend_midday_colum=check_all_characters_present(wkend_df,midday_column_check)
wkend_evening_column=check_all_characters_present(wkend_df,evening_column_check)
wkend_goal_column=check_all_characters_present(wkend_df,wkend_goal_column_check)
am_column,pm_column,midday_colum,evening_column,overall_goal_column,wkend_goal_column

(['AM Peak'],
 ['PM Peak'],
 ['Midday'],
 ['Evening'],
 ['Total Surveys'],
 ['Weekend Goals'])

In [44]:
wkend_new_df=pd.DataFrame()
wkend_new_df['ROUTE_SURVEYEDCode']=wkend_df['LS_NAME_CODE']
# new_df['CR_Pre_Early_AM']=cr_df[pre_early_am_column[0]]
# new_df['CR_Early_AM']=cr_df[early_am_column[0]]
wkend_new_df['CR_AM_Peak']=wkend_df[wkend_am_column[0]]
wkend_new_df['CR_Midday']=wkend_df[wkend_midday_colum[0]]
wkend_new_df['CR_PM_Peak']=wkend_df[wkend_pm_column[0]]
wkend_new_df['CR_Evening']=wkend_df[wkend_evening_column[0]]
wkend_new_df['Weekend Goal']=wkend_df[wkend_goal_column[0]]

wkend_new_df.fillna(0,inplace=True)
wkend_new_df

,ROUTE_SURVEYEDCode,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,Weekend Goal
0,SAN_1_1_01,0,0,0,0,6.400
1,SAN_1_1_00,0,0,0,0,0.000
2,SAN_1_2_01,0,0,0,0,4.475
3,SAN_1_2_00,0,0,0,0,0.000
4,SAN_1_3_01,0,0,0,0,5.100
5,SAN_1_3_00,0,0,0,0,0.000
6,SAN_1_4_01,0,0,0,0,11.600
7,SAN_1_4_00,0,0,0,0,0.000
8,SAN_1_5_01,0,0,0,0,21.850
9,SAN_1_5_00,0,0,0,0,0.000


In [10]:
new_df=pd.DataFrame()
new_df['ROUTE_SURVEYEDCode']=cr_df['LS_NAME_CODE']
# new_df['CR_Pre_Early_AM']=cr_df[pre_early_am_column[0]]
# new_df['CR_Early_AM']=cr_df[early_am_column[0]]
new_df['CR_AM_Peak']=cr_df[am_column[0]]
new_df['CR_Midday']=cr_df[midday_colum[0]]
new_df['CR_PM_Peak']=cr_df[pm_column[0]]
new_df['CR_Evening']=cr_df[evening_column[0]]
new_df['Overall Goal']=cr_df[overall_goal_column[0]]

new_df.fillna(0,inplace=True)
new_df

,ROUTE_SURVEYEDCode,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,Overall Goal
0,SAN_1_1_01,1.896765,2.592843,1.476375,1.071286,29.874778
1,SAN_1_1_00,0.610762,1.257072,1.533729,0.625901,0.000000
2,SAN_1_2_01,1.597572,1.280851,1.016962,0.622157,22.145837
3,SAN_1_2_00,0.696928,1.017533,1.418951,0.551209,0.000000
4,SAN_1_3_01,0.553204,0.676035,0.894075,0.884067,16.426045
5,SAN_1_3_00,1.079804,0.939968,0.597340,0.459227,0.000000
6,SAN_1_4_01,1.690795,3.530684,2.373890,1.207284,44.570096
7,SAN_1_4_00,0.969202,2.955031,2.385797,1.394760,0.000000
8,SAN_1_5_01,4.437076,6.241646,4.088108,4.109591,101.704011
9,SAN_1_5_00,4.548079,6.970255,4.649874,2.623523,0.000000


In [45]:
for index, row in wkend_new_df.iterrows():
    route_code = row['ROUTE_SURVEYEDCode']
    
    # Define a function to get the counts and IDs
    def get_counts_and_ids(time_values):
        subset_df = df[(df['ROUTE_SURVEYEDCode'] == route_code) & (df[time_column[0]].isin(time_values))]
        count = subset_df.shape[0]
        ids = subset_df['id'].values
        return count, ids
    
    # Calculate counts and IDs for each time slot
    am_value, am_value_ids = get_counts_and_ids(am_values)
    midday_value, midday_value_ids = get_counts_and_ids(midday_values)
    pm_value, pm_value_ids = get_counts_and_ids(pm_values)
    evening_value, evening_value_ids = get_counts_and_ids(evening_values)
    
    # Assign values to new_df
    new_df.loc[index, 'CR_Total'] = row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
    new_df.loc[index, 'DB_AM_Peak'] = am_value
    new_df.loc[index, 'DB_Midday'] = midday_value
    new_df.loc[index, 'DB_PM_Peak'] = pm_value
    new_df.loc[index, 'DB_Evening'] = evening_value
    new_df.loc[index, 'DB_Total'] = evening_value + am_value + midday_value + pm_value
    
    # Join the IDs as a comma-separated string
    new_df.loc[index, 'DB_AM_IDS'] = ', '.join(map(str, am_value_ids))
    new_df.loc[index, 'DB_Midday_IDS'] = ', '.join(map(str, midday_value_ids))
    new_df.loc[index, 'DB_PM_IDS'] = ', '.join(map(str, pm_value_ids))
    new_df.loc[index, 'DB_Evening_IDS'] = ', '.join(map(str, evening_value_ids))

In [11]:
for index, row in new_df.iterrows():
    route_code = row['ROUTE_SURVEYEDCode']
    
    # Define a function to get the counts and IDs
    def get_counts_and_ids(time_values):
        subset_df = df[(df['ROUTE_SURVEYEDCode'] == route_code) & (df[time_column[0]].isin(time_values))]
        count = subset_df.shape[0]
        ids = subset_df['id'].values
        return count, ids
    
    # Calculate counts and IDs for each time slot
    am_value, am_value_ids = get_counts_and_ids(am_values)
    midday_value, midday_value_ids = get_counts_and_ids(midday_values)
    pm_value, pm_value_ids = get_counts_and_ids(pm_values)
    evening_value, evening_value_ids = get_counts_and_ids(evening_values)
    
    # Assign values to new_df
    new_df.loc[index, 'CR_Total'] = row['CR_AM_Peak'] + row['CR_Midday'] + row['CR_PM_Peak'] + row['CR_Evening']
    new_df.loc[index, 'DB_AM_Peak'] = am_value
    new_df.loc[index, 'DB_Midday'] = midday_value
    new_df.loc[index, 'DB_PM_Peak'] = pm_value
    new_df.loc[index, 'DB_Evening'] = evening_value
    new_df.loc[index, 'DB_Total'] = evening_value + am_value + midday_value + pm_value
    
    # Join the IDs as a comma-separated string
    new_df.loc[index, 'DB_AM_IDS'] = ', '.join(map(str, am_value_ids))
    new_df.loc[index, 'DB_Midday_IDS'] = ', '.join(map(str, midday_value_ids))
    new_df.loc[index, 'DB_PM_IDS'] = ', '.join(map(str, pm_value_ids))
    new_df.loc[index, 'DB_Evening_IDS'] = ', '.join(map(str, evening_value_ids))

In [12]:
new_df

,ROUTE_SURVEYEDCode,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,Overall Goal,CR_Total,DB_AM_Peak,DB_Midday,DB_PM_Peak,DB_Evening,DB_Total,DB_AM_IDS,DB_Midday_IDS,DB_PM_IDS,DB_Evening_IDS
0,SAN_1_1_01,1.896765,2.592843,1.476375,1.071286,29.874778,7.037268,0.0,8.0,8.0,5.0,21.0,,"5105, 5157, 5273, 5275, 5276, 5277, 5290, 5291","5127, 5292, 5301, 5302, 5311, 5312, 5314, 5795","5321, 5323, 5324, 5325, 5326"
1,SAN_1_1_00,0.610762,1.257072,1.533729,0.625901,0.000000,4.027464,0.0,2.0,7.0,2.0,11.0,,"5278, 5287","5307, 5308, 5309, 5310, 5317, 5319, 5320","5327, 5328"
2,SAN_1_2_01,1.597572,1.280851,1.016962,0.622157,22.145837,4.517541,0.0,9.0,6.0,1.0,16.0,,"5107, 5148, 5611, 5679, 5800, 5835, 5916, 5935...","5158, 6003, 6004, 6006, 6008, 6009",5967
3,SAN_1_2_00,0.696928,1.017533,1.418951,0.551209,0.000000,3.684621,0.0,9.0,0.0,1.0,10.0,,"5143, 5841, 5842, 5919, 5921, 5922, 5923, 5942...",,5966
4,SAN_1_3_01,0.553204,0.676035,0.894075,0.884067,16.426045,3.007381,0.0,4.0,3.0,0.0,7.0,,"5819, 5993, 5994, 5995","5848, 5849, 5850",
5,SAN_1_3_00,1.079804,0.939968,0.597340,0.459227,0.000000,3.076340,0.0,9.0,2.0,0.0,11.0,,"5043, 5052, 5054, 5151, 5439, 5441, 5628, 5827...","5128, 5901",
6,SAN_1_4_01,1.690795,3.530684,2.373890,1.207284,44.570096,8.802653,0.0,9.0,14.0,0.0,23.0,,"5109, 5110, 5626, 5634, 5970, 5972, 5973, 5985...","5121, 5125, 5515, 5517, 5518, 5651, 5652, 5653...",
7,SAN_1_4_00,0.969202,2.955031,2.385797,1.394760,0.000000,7.704790,0.0,21.0,12.0,0.0,33.0,,"5104, 5111, 5616, 5617, 5627, 5629, 5630, 5632...","5124, 5512, 5514, 5646, 5647, 5952, 5955, 5956...",
8,SAN_1_5_01,4.437076,6.241646,4.088108,4.109591,101.704011,18.876421,4.0,30.0,14.0,4.0,52.0,"5058, 5059, 5060, 5522","5073, 5074, 5100, 5530, 5532, 5546, 5548, 5549...","5123, 5170, 5478, 5566, 5567, 5568, 5569, 5570...","5592, 5593, 5594, 5766"
9,SAN_1_5_00,4.548079,6.970255,4.649874,2.623523,0.000000,18.791732,0.0,31.0,12.0,4.0,47.0,,"5062, 5068, 5069, 5075, 5076, 5459, 5460, 5461...","5240, 5241, 5243, 5246, 5247, 5479, 5480, 5481...","5588, 5589, 5590, 5591"


In [13]:
new_df['ROUTE_SURVEYEDCode_Splited']=new_df['ROUTE_SURVEYEDCode'].apply(lambda x:('_').join(x.split('_')[:-1]) )

In [14]:
route_level_df=pd.DataFrame()

unique_routes=new_df['ROUTE_SURVEYEDCode_Splited'].unique()

route_level_df['ROUTE_SURVEYEDCode']=unique_routes

In [15]:
for index , row in route_level_df.iterrows():
    subset_df=new_df[new_df['ROUTE_SURVEYEDCode_Splited']==row['ROUTE_SURVEYEDCode']]
    # sum_per_route_cr = subset_df[['CR_Pre_Early_AM','CR_Early_AM','CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total']].sum()
    sum_per_route_cr = subset_df[['CR_AM_Peak', 'CR_Midday', 'CR_PM_Peak', 'CR_Evening','CR_Total','Overall Goal']].sum()
    sum_per_route_db = subset_df[['DB_AM_Peak', 'DB_Midday', 'DB_PM_Peak', 'DB_Evening','DB_Total']].sum()
    
    # route_level_df.loc[index,'CR_Pre_Early_AM']=sum_per_route_cr['CR_Pre_Early_AM']
    # route_level_df.loc[index,'CR_Early_AM']=sum_per_route_cr['CR_Early_AM']
    route_level_df.loc[index,'CR_AM_Peak']=sum_per_route_cr['CR_AM_Peak']
    route_level_df.loc[index,'CR_Midday']=sum_per_route_cr['CR_Midday']
    route_level_df.loc[index,'CR_PM_Peak']=sum_per_route_cr['CR_PM_Peak']
    route_level_df.loc[index,'CR_Evening']=sum_per_route_cr['CR_Evening']
    route_level_df.loc[index,'CR_Total']=sum_per_route_cr['CR_Total']
    route_level_df.loc[index,'CR_Overall_Goal']=sum_per_route_cr['Overall Goal']
    
    route_level_df.loc[index,'DB_AM_Peak']=sum_per_route_db['DB_AM_Peak']
    route_level_df.loc[index,'DB_Midday']=sum_per_route_db['DB_Midday']
    route_level_df.loc[index,'DB_PM_Peak']=sum_per_route_db['DB_PM_Peak']
    route_level_df.loc[index,'DB_Evening']=sum_per_route_db['DB_Evening']
    route_level_df.loc[index,'DB_Total']=sum_per_route_db['DB_Total']   
    route_level_df.loc[index,'DB_AM_IDS']=', '.join(str(value) for value in subset_df['DB_AM_IDS'].values)    
    route_level_df.loc[index,'DB_Midday_IDS']=', '.join(str(value) for value in subset_df['DB_Midday_IDS'].values)    
    route_level_df.loc[index,'DB_PM_IDS']=', '.join(str(value) for value in subset_df['DB_PM_IDS'].values)    
    route_level_df.loc[index,'DB_Evening_IDS']=', '.join(str(value) for value in subset_df['DB_Evening_IDS'].values)

In [16]:
route_level_df

,ROUTE_SURVEYEDCode,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,CR_Overall_Goal,DB_AM_Peak,DB_Midday,DB_PM_Peak,DB_Evening,DB_Total,DB_AM_IDS,DB_Midday_IDS,DB_PM_IDS,DB_Evening_IDS
0,SAN_1_1,2.507527,3.849915,3.010104,1.697187,11.064733,29.874778,0.0,10.0,15.0,7.0,32.0,",","5105, 5157, 5273, 5275, 5276, 5277, 5290, 5291...","5127, 5292, 5301, 5302, 5311, 5312, 5314, 5795...","5321, 5323, 5324, 5325, 5326, 5327, 5328"
1,SAN_1_2,2.294500,2.298384,2.435913,1.173366,8.202162,22.145837,0.0,18.0,6.0,2.0,26.0,",","5107, 5148, 5611, 5679, 5800, 5835, 5916, 5935...","5158, 6003, 6004, 6006, 6008, 6009,","5967, 5966"
2,SAN_1_3,1.633008,1.616003,1.491416,1.343294,6.083720,16.426045,0.0,13.0,5.0,0.0,18.0,",","5819, 5993, 5994, 5995, 5043, 5052, 5054, 5151...","5848, 5849, 5850, 5128, 5901",","
3,SAN_1_4,2.659997,6.485715,4.759687,2.602044,16.507443,44.570096,0.0,30.0,26.0,0.0,56.0,",","5109, 5110, 5626, 5634, 5970, 5972, 5973, 5985...","5121, 5125, 5515, 5517, 5518, 5651, 5652, 5653...",","
4,SAN_1_5,8.985155,13.211901,8.737982,6.733114,37.668152,101.704011,4.0,61.0,26.0,8.0,99.0,"5058, 5059, 5060, 5522,","5073, 5074, 5100, 5530, 5532, 5546, 5548, 5549...","5123, 5170, 5478, 5566, 5567, 5568, 5569, 5570...","5592, 5593, 5594, 5766, 5588, 5589, 5590, 5591"
5,SAN_1_6,12.101644,16.930776,11.608637,7.856310,48.497367,130.942892,2.0,63.0,48.0,13.0,126.0,", 5668, 5670","5064, 5065, 5066, 5067, 5077, 5106, 5155, 5210...","5180, 5249, 5250, 5482, 5484, 5637, 5640, 5641...","5254, 5255, 5256, 5257, 5258, 5485, 5487, 5488..."
6,SAN_1_7,0.682796,2.341270,1.415940,0.810992,5.250997,14.177693,0.0,16.0,3.0,1.0,20.0,",","5044, 5045, 5050, 5051, 5053, 5055, 5056, 5057...","5903, 5129, 5856",", 5126"
7,SAN_1_12,7.779806,9.736701,8.874679,5.392142,31.783328,85.814986,2.0,43.0,27.0,15.0,87.0,"5369, 5601,","5150, 5152, 5383, 5400, 5403, 5404, 5406, 5445...","5120, 5182, 5419, 5420, 5426, 5428, 5430, 5477...","5597, 5655, 5658, 5433, 5435, 5436, 5660, 5661..."
8,SAN_1_14,2.200258,6.945632,5.178756,1.772019,16.096664,43.460994,0.0,23.0,21.0,3.0,47.0,",","5009, 5010, 5020, 5021, 5022, 5341, 5342, 5344...","5023, 5024, 5030, 5031, 5032, 5033, 5101, 5356...","5909, 5908, 5910"
9,SAN_1_757,2.890908,3.116570,2.369358,1.856307,10.233142,27.629485,0.0,30.0,5.0,8.0,43.0,",","5085, 5086, 5089, 5090, 5135, 5136, 5563, 5696...","5579, 5580, 5117, 5159, 5581",", 5134, 5583, 5656, 5767, 6023, 6024, 6025, 6027"


In [17]:
for index, row in route_level_df.iterrows():
    am_peak_diff=row['CR_AM_Peak']-row['DB_AM_Peak']
    midday_diff=row['CR_Midday']-row['DB_Midday']    
    pm_peak_diff=row['CR_PM_Peak']-row['DB_PM_Peak']
    evening_diff=row['CR_Evening']-row['DB_Evening']
    total_diff=row['CR_Total']-row['DB_Total']
    overall_difference=row['CR_Overall_Goal']-row['DB_Total']
    route_level_df.loc[index, 'AM_DIFFERENCE'] = math.ceil(max(0, am_peak_diff))
    route_level_df.loc[index, 'Midday_DIFFERENCE'] = math.ceil(max(0, midday_diff))
    route_level_df.loc[index, 'PM_DIFFERENCE'] = math.ceil(max(0, pm_peak_diff))
    route_level_df.loc[index, 'Evening_DIFFERENCE'] = math.ceil(max(0, evening_diff))
    # route_level_df.loc[index, 'Total_DIFFERENCE'] = math.ceil(max(0, total_diff))
    route_level_df.loc[index, 'Total_DIFFERENCE'] =math.ceil(max(0, am_peak_diff))+math.ceil(max(0, midday_diff))+math.ceil(max(0, pm_peak_diff))+math.ceil(max(0, evening_diff))
    route_level_df.loc[index, 'Overall_Goal_DIfference'] = math.ceil(max(0,overall_difference))
route_level_df

,ROUTE_SURVEYEDCode,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,CR_Overall_Goal,DB_AM_Peak,DB_Midday,DB_PM_Peak,...,DB_AM_IDS,DB_Midday_IDS,DB_PM_IDS,DB_Evening_IDS,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE,Overall_Goal_DIfference
0,SAN_1_1,2.507527,3.849915,3.010104,1.697187,11.064733,29.874778,0.0,10.0,15.0,...,",","5105, 5157, 5273, 5275, 5276, 5277, 5290, 5291...","5127, 5292, 5301, 5302, 5311, 5312, 5314, 5795...","5321, 5323, 5324, 5325, 5326, 5327, 5328",3.0,0.0,0.0,0.0,3.0,0.0
1,SAN_1_2,2.294500,2.298384,2.435913,1.173366,8.202162,22.145837,0.0,18.0,6.0,...,",","5107, 5148, 5611, 5679, 5800, 5835, 5916, 5935...","5158, 6003, 6004, 6006, 6008, 6009,","5967, 5966",3.0,0.0,0.0,0.0,3.0,0.0
2,SAN_1_3,1.633008,1.616003,1.491416,1.343294,6.083720,16.426045,0.0,13.0,5.0,...,",","5819, 5993, 5994, 5995, 5043, 5052, 5054, 5151...","5848, 5849, 5850, 5128, 5901",",",2.0,0.0,0.0,2.0,4.0,0.0
3,SAN_1_4,2.659997,6.485715,4.759687,2.602044,16.507443,44.570096,0.0,30.0,26.0,...,",","5109, 5110, 5626, 5634, 5970, 5972, 5973, 5985...","5121, 5125, 5515, 5517, 5518, 5651, 5652, 5653...",",",3.0,0.0,0.0,3.0,6.0,0.0
4,SAN_1_5,8.985155,13.211901,8.737982,6.733114,37.668152,101.704011,4.0,61.0,26.0,...,"5058, 5059, 5060, 5522,","5073, 5074, 5100, 5530, 5532, 5546, 5548, 5549...","5123, 5170, 5478, 5566, 5567, 5568, 5569, 5570...","5592, 5593, 5594, 5766, 5588, 5589, 5590, 5591",5.0,0.0,0.0,0.0,5.0,3.0
5,SAN_1_6,12.101644,16.930776,11.608637,7.856310,48.497367,130.942892,2.0,63.0,48.0,...,", 5668, 5670","5064, 5065, 5066, 5067, 5077, 5106, 5155, 5210...","5180, 5249, 5250, 5482, 5484, 5637, 5640, 5641...","5254, 5255, 5256, 5257, 5258, 5485, 5487, 5488...",11.0,0.0,0.0,0.0,11.0,5.0
6,SAN_1_7,0.682796,2.341270,1.415940,0.810992,5.250997,14.177693,0.0,16.0,3.0,...,",","5044, 5045, 5050, 5051, 5053, 5055, 5056, 5057...","5903, 5129, 5856",", 5126",1.0,0.0,0.0,0.0,1.0,0.0
7,SAN_1_12,7.779806,9.736701,8.874679,5.392142,31.783328,85.814986,2.0,43.0,27.0,...,"5369, 5601,","5150, 5152, 5383, 5400, 5403, 5404, 5406, 5445...","5120, 5182, 5419, 5420, 5426, 5428, 5430, 5477...","5597, 5655, 5658, 5433, 5435, 5436, 5660, 5661...",6.0,0.0,0.0,0.0,6.0,0.0
8,SAN_1_14,2.200258,6.945632,5.178756,1.772019,16.096664,43.460994,0.0,23.0,21.0,...,",","5009, 5010, 5020, 5021, 5022, 5341, 5342, 5344...","5023, 5024, 5030, 5031, 5032, 5033, 5101, 5356...","5909, 5908, 5910",3.0,0.0,0.0,0.0,3.0,0.0
9,SAN_1_757,2.890908,3.116570,2.369358,1.856307,10.233142,27.629485,0.0,30.0,5.0,...,",","5085, 5086, 5089, 5090, 5135, 5136, 5563, 5696...","5579, 5580, 5117, 5159, 5581",", 5134, 5583, 5656, 5767, 6023, 6024, 6025, 6027",3.0,0.0,0.0,0.0,3.0,0.0


In [18]:
route_level_df['Overall_Goal_DIfference'].sum()

14.0

In [19]:
comparison_df=copy.deepcopy(route_level_df)
new_route_level_df=copy.deepcopy(route_level_df)

In [20]:
new_route_level_df

,ROUTE_SURVEYEDCode,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,CR_Overall_Goal,DB_AM_Peak,DB_Midday,DB_PM_Peak,...,DB_AM_IDS,DB_Midday_IDS,DB_PM_IDS,DB_Evening_IDS,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE,Overall_Goal_DIfference
0,SAN_1_1,2.507527,3.849915,3.010104,1.697187,11.064733,29.874778,0.0,10.0,15.0,...,",","5105, 5157, 5273, 5275, 5276, 5277, 5290, 5291...","5127, 5292, 5301, 5302, 5311, 5312, 5314, 5795...","5321, 5323, 5324, 5325, 5326, 5327, 5328",3.0,0.0,0.0,0.0,3.0,0.0
1,SAN_1_2,2.294500,2.298384,2.435913,1.173366,8.202162,22.145837,0.0,18.0,6.0,...,",","5107, 5148, 5611, 5679, 5800, 5835, 5916, 5935...","5158, 6003, 6004, 6006, 6008, 6009,","5967, 5966",3.0,0.0,0.0,0.0,3.0,0.0
2,SAN_1_3,1.633008,1.616003,1.491416,1.343294,6.083720,16.426045,0.0,13.0,5.0,...,",","5819, 5993, 5994, 5995, 5043, 5052, 5054, 5151...","5848, 5849, 5850, 5128, 5901",",",2.0,0.0,0.0,2.0,4.0,0.0
3,SAN_1_4,2.659997,6.485715,4.759687,2.602044,16.507443,44.570096,0.0,30.0,26.0,...,",","5109, 5110, 5626, 5634, 5970, 5972, 5973, 5985...","5121, 5125, 5515, 5517, 5518, 5651, 5652, 5653...",",",3.0,0.0,0.0,3.0,6.0,0.0
4,SAN_1_5,8.985155,13.211901,8.737982,6.733114,37.668152,101.704011,4.0,61.0,26.0,...,"5058, 5059, 5060, 5522,","5073, 5074, 5100, 5530, 5532, 5546, 5548, 5549...","5123, 5170, 5478, 5566, 5567, 5568, 5569, 5570...","5592, 5593, 5594, 5766, 5588, 5589, 5590, 5591",5.0,0.0,0.0,0.0,5.0,3.0
5,SAN_1_6,12.101644,16.930776,11.608637,7.856310,48.497367,130.942892,2.0,63.0,48.0,...,", 5668, 5670","5064, 5065, 5066, 5067, 5077, 5106, 5155, 5210...","5180, 5249, 5250, 5482, 5484, 5637, 5640, 5641...","5254, 5255, 5256, 5257, 5258, 5485, 5487, 5488...",11.0,0.0,0.0,0.0,11.0,5.0
6,SAN_1_7,0.682796,2.341270,1.415940,0.810992,5.250997,14.177693,0.0,16.0,3.0,...,",","5044, 5045, 5050, 5051, 5053, 5055, 5056, 5057...","5903, 5129, 5856",", 5126",1.0,0.0,0.0,0.0,1.0,0.0
7,SAN_1_12,7.779806,9.736701,8.874679,5.392142,31.783328,85.814986,2.0,43.0,27.0,...,"5369, 5601,","5150, 5152, 5383, 5400, 5403, 5404, 5406, 5445...","5120, 5182, 5419, 5420, 5426, 5428, 5430, 5477...","5597, 5655, 5658, 5433, 5435, 5436, 5660, 5661...",6.0,0.0,0.0,0.0,6.0,0.0
8,SAN_1_14,2.200258,6.945632,5.178756,1.772019,16.096664,43.460994,0.0,23.0,21.0,...,",","5009, 5010, 5020, 5021, 5022, 5341, 5342, 5344...","5023, 5024, 5030, 5031, 5032, 5033, 5101, 5356...","5909, 5908, 5910",3.0,0.0,0.0,0.0,3.0,0.0
9,SAN_1_757,2.890908,3.116570,2.369358,1.856307,10.233142,27.629485,0.0,30.0,5.0,...,",","5085, 5086, 5089, 5090, 5135, 5136, 5563, 5696...","5579, 5580, 5117, 5159, 5581",", 5134, 5583, 5656, 5767, 6023, 6024, 6025, 6027",3.0,0.0,0.0,0.0,3.0,0.0


In [21]:
comparison_df

,ROUTE_SURVEYEDCode,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,CR_Overall_Goal,DB_AM_Peak,DB_Midday,DB_PM_Peak,...,DB_AM_IDS,DB_Midday_IDS,DB_PM_IDS,DB_Evening_IDS,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE,Overall_Goal_DIfference
0,SAN_1_1,2.507527,3.849915,3.010104,1.697187,11.064733,29.874778,0.0,10.0,15.0,...,",","5105, 5157, 5273, 5275, 5276, 5277, 5290, 5291...","5127, 5292, 5301, 5302, 5311, 5312, 5314, 5795...","5321, 5323, 5324, 5325, 5326, 5327, 5328",3.0,0.0,0.0,0.0,3.0,0.0
1,SAN_1_2,2.294500,2.298384,2.435913,1.173366,8.202162,22.145837,0.0,18.0,6.0,...,",","5107, 5148, 5611, 5679, 5800, 5835, 5916, 5935...","5158, 6003, 6004, 6006, 6008, 6009,","5967, 5966",3.0,0.0,0.0,0.0,3.0,0.0
2,SAN_1_3,1.633008,1.616003,1.491416,1.343294,6.083720,16.426045,0.0,13.0,5.0,...,",","5819, 5993, 5994, 5995, 5043, 5052, 5054, 5151...","5848, 5849, 5850, 5128, 5901",",",2.0,0.0,0.0,2.0,4.0,0.0
3,SAN_1_4,2.659997,6.485715,4.759687,2.602044,16.507443,44.570096,0.0,30.0,26.0,...,",","5109, 5110, 5626, 5634, 5970, 5972, 5973, 5985...","5121, 5125, 5515, 5517, 5518, 5651, 5652, 5653...",",",3.0,0.0,0.0,3.0,6.0,0.0
4,SAN_1_5,8.985155,13.211901,8.737982,6.733114,37.668152,101.704011,4.0,61.0,26.0,...,"5058, 5059, 5060, 5522,","5073, 5074, 5100, 5530, 5532, 5546, 5548, 5549...","5123, 5170, 5478, 5566, 5567, 5568, 5569, 5570...","5592, 5593, 5594, 5766, 5588, 5589, 5590, 5591",5.0,0.0,0.0,0.0,5.0,3.0
5,SAN_1_6,12.101644,16.930776,11.608637,7.856310,48.497367,130.942892,2.0,63.0,48.0,...,", 5668, 5670","5064, 5065, 5066, 5067, 5077, 5106, 5155, 5210...","5180, 5249, 5250, 5482, 5484, 5637, 5640, 5641...","5254, 5255, 5256, 5257, 5258, 5485, 5487, 5488...",11.0,0.0,0.0,0.0,11.0,5.0
6,SAN_1_7,0.682796,2.341270,1.415940,0.810992,5.250997,14.177693,0.0,16.0,3.0,...,",","5044, 5045, 5050, 5051, 5053, 5055, 5056, 5057...","5903, 5129, 5856",", 5126",1.0,0.0,0.0,0.0,1.0,0.0
7,SAN_1_12,7.779806,9.736701,8.874679,5.392142,31.783328,85.814986,2.0,43.0,27.0,...,"5369, 5601,","5150, 5152, 5383, 5400, 5403, 5404, 5406, 5445...","5120, 5182, 5419, 5420, 5426, 5428, 5430, 5477...","5597, 5655, 5658, 5433, 5435, 5436, 5660, 5661...",6.0,0.0,0.0,0.0,6.0,0.0
8,SAN_1_14,2.200258,6.945632,5.178756,1.772019,16.096664,43.460994,0.0,23.0,21.0,...,",","5009, 5010, 5020, 5021, 5022, 5341, 5342, 5344...","5023, 5024, 5030, 5031, 5032, 5033, 5101, 5356...","5909, 5908, 5910",3.0,0.0,0.0,0.0,3.0,0.0
9,SAN_1_757,2.890908,3.116570,2.369358,1.856307,10.233142,27.629485,0.0,30.0,5.0,...,",","5085, 5086, 5089, 5090, 5135, 5136, 5563, 5696...","5579, 5580, 5117, 5159, 5581",", 5134, 5583, 5656, 5767, 6023, 6024, 6025, 6027",3.0,0.0,0.0,0.0,3.0,0.0


In [22]:
for index , row in comparison_df.iterrows():
    comparison_df.loc[index,'Total_DIFFERENCE']=math.ceil(max(0,(row['CR_Total']-row['DB_Total'])))
comparison_df

,ROUTE_SURVEYEDCode,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,CR_Overall_Goal,DB_AM_Peak,DB_Midday,DB_PM_Peak,...,DB_AM_IDS,DB_Midday_IDS,DB_PM_IDS,DB_Evening_IDS,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE,Overall_Goal_DIfference
0,SAN_1_1,2.507527,3.849915,3.010104,1.697187,11.064733,29.874778,0.0,10.0,15.0,...,",","5105, 5157, 5273, 5275, 5276, 5277, 5290, 5291...","5127, 5292, 5301, 5302, 5311, 5312, 5314, 5795...","5321, 5323, 5324, 5325, 5326, 5327, 5328",3.0,0.0,0.0,0.0,0.0,0.0
1,SAN_1_2,2.294500,2.298384,2.435913,1.173366,8.202162,22.145837,0.0,18.0,6.0,...,",","5107, 5148, 5611, 5679, 5800, 5835, 5916, 5935...","5158, 6003, 6004, 6006, 6008, 6009,","5967, 5966",3.0,0.0,0.0,0.0,0.0,0.0
2,SAN_1_3,1.633008,1.616003,1.491416,1.343294,6.083720,16.426045,0.0,13.0,5.0,...,",","5819, 5993, 5994, 5995, 5043, 5052, 5054, 5151...","5848, 5849, 5850, 5128, 5901",",",2.0,0.0,0.0,2.0,0.0,0.0
3,SAN_1_4,2.659997,6.485715,4.759687,2.602044,16.507443,44.570096,0.0,30.0,26.0,...,",","5109, 5110, 5626, 5634, 5970, 5972, 5973, 5985...","5121, 5125, 5515, 5517, 5518, 5651, 5652, 5653...",",",3.0,0.0,0.0,3.0,0.0,0.0
4,SAN_1_5,8.985155,13.211901,8.737982,6.733114,37.668152,101.704011,4.0,61.0,26.0,...,"5058, 5059, 5060, 5522,","5073, 5074, 5100, 5530, 5532, 5546, 5548, 5549...","5123, 5170, 5478, 5566, 5567, 5568, 5569, 5570...","5592, 5593, 5594, 5766, 5588, 5589, 5590, 5591",5.0,0.0,0.0,0.0,0.0,3.0
5,SAN_1_6,12.101644,16.930776,11.608637,7.856310,48.497367,130.942892,2.0,63.0,48.0,...,", 5668, 5670","5064, 5065, 5066, 5067, 5077, 5106, 5155, 5210...","5180, 5249, 5250, 5482, 5484, 5637, 5640, 5641...","5254, 5255, 5256, 5257, 5258, 5485, 5487, 5488...",11.0,0.0,0.0,0.0,0.0,5.0
6,SAN_1_7,0.682796,2.341270,1.415940,0.810992,5.250997,14.177693,0.0,16.0,3.0,...,",","5044, 5045, 5050, 5051, 5053, 5055, 5056, 5057...","5903, 5129, 5856",", 5126",1.0,0.0,0.0,0.0,0.0,0.0
7,SAN_1_12,7.779806,9.736701,8.874679,5.392142,31.783328,85.814986,2.0,43.0,27.0,...,"5369, 5601,","5150, 5152, 5383, 5400, 5403, 5404, 5406, 5445...","5120, 5182, 5419, 5420, 5426, 5428, 5430, 5477...","5597, 5655, 5658, 5433, 5435, 5436, 5660, 5661...",6.0,0.0,0.0,0.0,0.0,0.0
8,SAN_1_14,2.200258,6.945632,5.178756,1.772019,16.096664,43.460994,0.0,23.0,21.0,...,",","5009, 5010, 5020, 5021, 5022, 5341, 5342, 5344...","5023, 5024, 5030, 5031, 5032, 5033, 5101, 5356...","5909, 5908, 5910",3.0,0.0,0.0,0.0,0.0,0.0
9,SAN_1_757,2.890908,3.116570,2.369358,1.856307,10.233142,27.629485,0.0,30.0,5.0,...,",","5085, 5086, 5089, 5090, 5135, 5136, 5563, 5696...","5579, 5580, 5117, 5159, 5581",", 5134, 5583, 5656, 5767, 6023, 6024, 6025, 6027",3.0,0.0,0.0,0.0,0.0,0.0


In [23]:
trip_oppo_dir_column_check=['tripinoppodir']
trip_oppo_dir_column=check_all_characters_present(df,trip_oppo_dir_column_check)

route_survey_name_column_check=['routesurveyed']
route_survey_name_column=check_all_characters_present(df,route_survey_name_column_check)

oppo_dir_time_column_check=['oppodirtriptimecode']
oppo_dir_time_column=check_all_characters_present(df,oppo_dir_time_column_check)

trip_code_column_check=['prevtransferscode','nexttransferscode']
trip_code_column=check_all_characters_present(df,trip_code_column_check)
trip_code_column.sort()

prev_trip_route_code_column_check=['tripfirstroutecode','tripsecondroutecode','tripthirdroutecode','tripfourthroutecode']
next_trip_route_code_column_check=['tripnextroutecode','tripafterroutecode','trip3rdroutecode','triplast4thrtecode']
prev_trip_route_code_column=check_all_characters_present(df,prev_trip_route_code_column_check)
next_trip_route_code_column=check_all_characters_present(df,next_trip_route_code_column_check)
prev_trip_route_code_column,next_trip_route_code_column,trip_code_column

(['TRIP_FIRST_ROUTECode',
  'TRIP_SECOND_ROUTECode',
  'TRIP_THIRD_ROUTECode',
  'TRIP_FOURTH_ROUTECode'],
 ['TRIP_NEXT_ROUTECode',
  'TRIP_AFTER_ROUTECode',
  'TRIP_3RD_ROUTECode',
  'TRIP_LAST4TH_RTECode'],
 ['NEXT_TRANSFERSCode', 'PREV_TRANSFERSCode'])

In [24]:
values_to_replace = ['-oth-']
df[[*prev_trip_route_code_column, *next_trip_route_code_column]] = df[
    [*prev_trip_route_code_column, *next_trip_route_code_column]
].replace(values_to_replace, np.nan)

In [25]:
reverse_df=df[df[trip_oppo_dir_column[0]].str.lower()=='yes'][['id',*route_survey_column,*route_survey_name_column,*trip_code_column,*prev_trip_route_code_column,*next_trip_route_code_column]]

reverse_df[route_survey_column[0]]=reverse_df[route_survey_column[0]].apply(lambda x: '_'.join(x.split("_")[:-1]))

reverse_df.reset_index(inplace=True,drop=True)

# reverse_df[[*prev_trip_route_code_column,*next_trip_route_code_column]].fillna('',inplace=True)
reverse_df.fillna('',inplace=True)

In [26]:
reverse_df[reverse_df[route_survey_column[0]]=='SAN_1_1']

,id,ROUTE_SURVEYEDCode,ROUTE_SURVEYED,NEXT_TRANSFERSCode,PREV_TRANSFERSCode,TRIP_FIRST_ROUTECode,TRIP_SECOND_ROUTECode,TRIP_THIRD_ROUTECode,TRIP_FOURTH_ROUTECode,TRIP_NEXT_ROUTECode,TRIP_AFTER_ROUTECode,TRIP_3RD_ROUTECode,TRIP_LAST4TH_RTECode
45,5105,SAN_1_1,1 - Outbound - To Castaic,0.0,1.0,SAN_1_12,,,,,,,
47,5127,SAN_1_1,1 - Outbound - To Castaic,0.0,1.0,SAN_1_12,,,,,,,
93,5273,SAN_1_1,1 - Outbound - To Castaic,0.0,1.0,SAN_1_6,,,,,,,
94,5275,SAN_1_1,1 - Outbound - To Castaic,0.0,0.0,,,,,,,,
95,5278,SAN_1_1,1 - Inbound - To McBean,1.0,0.0,,,,,SAN_1_5,,,
96,5281,SAN_1_1,1 - Outbound - To Castaic,0.0,1.0,SAN_1_5,,,,,,,
97,5283,SAN_1_1,1 - Outbound - To Castaic,0.0,0.0,,,,,,,,
98,5284,SAN_1_1,1 - Outbound - To Castaic,0.0,0.0,,,,,,,,
99,5285,SAN_1_1,1 - Outbound - To Castaic,0.0,1.0,SAN_1_6,,,,,,,
100,5287,SAN_1_1,1 - Inbound - To McBean,1.0,0.0,,,,,SAN_1_5,,,


In [27]:
random.choice([0,1])

1

In [28]:
route_level_df[route_level_df[route_survey_column[0]]=='SAN_1_5']['Overall_Goal_DIfference']

4    3.0
Name: Overall_Goal_DIfference, dtype: float64

In [29]:
def get_valid_routes(row, route_code_column):
    result_array = reverse_df[reverse_df['id'] == row['id']][route_code_column].values
    values_in_list = result_array[0, :]
    return [value for value in values_in_list if not (pd.isna(value) or value == '')]

def process_route(route, counter_list, counter_prefix):
    counter_list[0] += 1
    rev_prefix=f'Rev-{counter_prefix}'
    random_choice = random.choice([counter_prefix,rev_prefix ])
    value = int(route_level_df[route_level_df[route_survey_column[0]] == route]['Total_DIFFERENCE'].values)  
    if value > 0:
        reverse_df.loc[index, 'Type'] = f'{random_choice}{counter_list[0]}'
        route_level_df.loc[route_level_df[route_survey_column[0]] == route, 'Total_DIFFERENCE'] = value - 1
        return True
    return False

In [30]:
for index, row in reverse_df.iterrows():
    value=int(route_level_df[route_level_df[route_survey_column[0]] == row[route_survey_column[0]]]['Total_DIFFERENCE'].values)
    overall_value=int(route_level_df[route_level_df[route_survey_column[0]] == row[route_survey_column[0]]]['Overall_Goal_DIfference'].values)
    prev_trans_value = int(reverse_df[reverse_df['id'] == row['id']][trip_code_column[1]].values)
    next_trans_value = int(reverse_df[reverse_df['id'] == row['id']][trip_code_column[0]].values)
    if value>0: 
        if random.choice([0,1]):
            reverse_df.loc[index,'Type']='Reverse'
            route_level_df.loc[route_level_df[route_survey_column[0]] == row[route_survey_column[0]], 'Total_DIFFERENCE'] = value - 1
            route_level_df.loc[route_level_df[route_survey_column[0]] == row[route_survey_column[0]], 'Overall_Goal_DIfference'] = overall_value - 1
        else:
            if prev_trans_value:
                result_array = reverse_df[reverse_df['id'] == row['id']][prev_trip_route_code_column].values
                values_in_list = result_array[0, :]
                valid_values = [value for value in values_in_list if not (pd.isna(value) or value == '')]
                prev_counter=0
                for route in valid_values:
                    prev_counter+=1
                    value=int(route_level_df[(route_level_df[route_survey_column[0]]==row[route_survey_column[0]])]['Total_DIFFERENCE'].values)
                    if value >0:
                        
                        reverse_df.loc[index,'Type']=f'{random.choice([f"p{prev_counter}",f"Rev-p{prev_counter}"])}'
                        route_level_df.loc[route_level_df[route_survey_column[0]] == row[route_survey_column[0]],'Total_DIFFERENCE'] = value - 1
                        route_level_df.loc[route_level_df[route_survey_column[0]] == row[route_survey_column[0]], 'Overall_Goal_DIfference'] = overall_value - 1
                        
                        break
            elif next_trans_value:
                result_array = reverse_df[reverse_df['id'] == row['id']][next_trip_route_code_column].values
                values_in_list = result_array[0, :]
                valid_values = [value for value in values_in_list if not (pd.isna(value) or value == '')]
                next_counter=0
                for route in valid_values:
                    next_counter+=1
                    value=int(route_level_df[(route_level_df[route_survey_column[0]]==row[route_survey_column[0]])]['Total_DIFFERENCE'].values)
                    if value >0:
                        reverse_df.loc[index,'Type']=f'{random.choice([f"n{next_counter}",f"Rev-n{next_counter}"])}'
                        route_level_df.loc[route_level_df[route_survey_column[0]] == row[route_survey_column[0]],'Total_DIFFERENCE'] = value - 1
                        route_level_df.loc[route_level_df[route_survey_column[0]] == row[route_survey_column[0]], 'Overall_Goal_DIfference'] = overall_value - 1
                        break
            else:
                reverse_df.loc[index,'Type']='Reverse'
                route_level_df.loc[route_level_df[route_survey_column[0]] == row[route_survey_column[0]],'Total_DIFFERENCE'] = value - 1
                route_level_df.loc[route_level_df[route_survey_column[0]] == row[route_survey_column[0]], 'Overall_Goal_DIfference'] = overall_value - 1
                
    elif overall_value>0:
        print("Working in Overall value")
        print(overall_value)
        reverse_df.loc[index,'Type']='Reverse'
        route_level_df.loc[route_level_df[route_survey_column[0]] == row[route_survey_column[0]], 'Overall_Goal_DIfference'] = overall_value - 1
    else:
        reverse_df.loc[index,'Type']=''


Working in Overall value
2


In [31]:
reverse_df['Type'].fillna('',inplace=True)

In [32]:
reverse_df[reverse_df['Type']!=''].shape

(57, 14)

In [33]:
# reverse_df.to_csv("TESTING_REVERSE.csv",index=False)

In [34]:
route_level_df

,ROUTE_SURVEYEDCode,CR_AM_Peak,CR_Midday,CR_PM_Peak,CR_Evening,CR_Total,CR_Overall_Goal,DB_AM_Peak,DB_Midday,DB_PM_Peak,...,DB_AM_IDS,DB_Midday_IDS,DB_PM_IDS,DB_Evening_IDS,AM_DIFFERENCE,Midday_DIFFERENCE,PM_DIFFERENCE,Evening_DIFFERENCE,Total_DIFFERENCE,Overall_Goal_DIfference
0,SAN_1_1,2.507527,3.849915,3.010104,1.697187,11.064733,29.874778,0.0,10.0,15.0,...,",","5105, 5157, 5273, 5275, 5276, 5277, 5290, 5291...","5127, 5292, 5301, 5302, 5311, 5312, 5314, 5795...","5321, 5323, 5324, 5325, 5326, 5327, 5328",3.0,0.0,0.0,0.0,0.0,-3.0
1,SAN_1_2,2.294500,2.298384,2.435913,1.173366,8.202162,22.145837,0.0,18.0,6.0,...,",","5107, 5148, 5611, 5679, 5800, 5835, 5916, 5935...","5158, 6003, 6004, 6006, 6008, 6009,","5967, 5966",3.0,0.0,0.0,0.0,0.0,-3.0
2,SAN_1_3,1.633008,1.616003,1.491416,1.343294,6.083720,16.426045,0.0,13.0,5.0,...,",","5819, 5993, 5994, 5995, 5043, 5052, 5054, 5151...","5848, 5849, 5850, 5128, 5901",",",2.0,0.0,0.0,2.0,0.0,-4.0
3,SAN_1_4,2.659997,6.485715,4.759687,2.602044,16.507443,44.570096,0.0,30.0,26.0,...,",","5109, 5110, 5626, 5634, 5970, 5972, 5973, 5985...","5121, 5125, 5515, 5517, 5518, 5651, 5652, 5653...",",",3.0,0.0,0.0,3.0,0.0,-6.0
4,SAN_1_5,8.985155,13.211901,8.737982,6.733114,37.668152,101.704011,4.0,61.0,26.0,...,"5058, 5059, 5060, 5522,","5073, 5074, 5100, 5530, 5532, 5546, 5548, 5549...","5123, 5170, 5478, 5566, 5567, 5568, 5569, 5570...","5592, 5593, 5594, 5766, 5588, 5589, 5590, 5591",5.0,0.0,0.0,0.0,0.0,-2.0
5,SAN_1_6,12.101644,16.930776,11.608637,7.856310,48.497367,130.942892,2.0,63.0,48.0,...,", 5668, 5670","5064, 5065, 5066, 5067, 5077, 5106, 5155, 5210...","5180, 5249, 5250, 5482, 5484, 5637, 5640, 5641...","5254, 5255, 5256, 5257, 5258, 5485, 5487, 5488...",11.0,0.0,0.0,0.0,0.0,-6.0
6,SAN_1_7,0.682796,2.341270,1.415940,0.810992,5.250997,14.177693,0.0,16.0,3.0,...,",","5044, 5045, 5050, 5051, 5053, 5055, 5056, 5057...","5903, 5129, 5856",", 5126",1.0,0.0,0.0,0.0,0.0,-1.0
7,SAN_1_12,7.779806,9.736701,8.874679,5.392142,31.783328,85.814986,2.0,43.0,27.0,...,"5369, 5601,","5150, 5152, 5383, 5400, 5403, 5404, 5406, 5445...","5120, 5182, 5419, 5420, 5426, 5428, 5430, 5477...","5597, 5655, 5658, 5433, 5435, 5436, 5660, 5661...",6.0,0.0,0.0,0.0,0.0,-6.0
8,SAN_1_14,2.200258,6.945632,5.178756,1.772019,16.096664,43.460994,0.0,23.0,21.0,...,",","5009, 5010, 5020, 5021, 5022, 5341, 5342, 5344...","5023, 5024, 5030, 5031, 5032, 5033, 5101, 5356...","5909, 5908, 5910",3.0,0.0,0.0,0.0,0.0,-3.0
9,SAN_1_757,2.890908,3.116570,2.369358,1.856307,10.233142,27.629485,0.0,30.0,5.0,...,",","5085, 5086, 5089, 5090, 5135, 5136, 5563, 5696...","5579, 5580, 5117, 5159, 5581",", 5134, 5583, 5656, 5767, 6023, 6024, 6025, 6027",3.0,0.0,0.0,0.0,0.0,-3.0


In [35]:
testets

NameError: name 'testets' is not defined

In [ ]:
route_level_df

In [ ]:

def get_valid_routes(row, route_code_column):
    result_array = reverse_df[reverse_df['id'] == row['id']][route_code_column].values
    values_in_list = result_array[0, :]
    return [value for value in values_in_list if not (pd.isna(value) or value == '')]

def process_route(route, counter_list, counter_prefix):
    counter_list[0] += 1
    rev_prefix=f'Rev-{counter_prefix}'
    random_choice = random.choice([counter_prefix,rev_prefix ])
    value = int(route_level_df[route_level_df[route_survey_column[0]] == route]['Total_DIFFERENCE'].values[0]) if len(route_level_df[route_level_df[route_survey_column[0]] == route]['Total_DIFFERENCE'].values) > 0 else 0  
    if value > 0:
        reverse_df.loc[index, 'Type'] = f'{random_choice}{counter_list[0]}'
        route_level_df.loc[route_level_df[route_survey_column[0]] == route, 'Total_DIFFERENCE'] = value - 1
        return True
    return False

# implemented the logic for handling Type values for all the data where  opossite direction is True
# for index, row in reverse_df.iterrows():
#     random_value = random.choice([0, 1])
#     value = int(route_level_df[route_level_df[route_survey_column[0]] == row[route_survey_column[0]]]['Total_DIFFERENCE'].values)
#     prev_trans_value = int(df[df['id'] == row['id']][trip_code_column[1]].values)
#     next_trans_value = int(df[df['id'] == row['id']][trip_code_column[0]].values)
#     counter = [0]  # Use a list to store the counter value

#     if random_value:
#         if value > 0:
#             reverse_df.loc[index, 'Type'] = 'Reverse'
#             route_level_df.loc[route_level_df[route_survey_column[0]] == row[route_survey_column[0]], 'Total_DIFFERENCE'] = value - 1
#         elif prev_trans_value:
#             for route in get_valid_routes(row, prev_trip_route_code_column):
#                 result_value=process_route(route, counter, 'p')
#                 if result_value:
#                     break
#                 else:
#                     reverse_df.loc[index, 'Type'] = f'{random.choice(["p1","Rev-p1"])}'
#                     route_level_df.loc[route_level_df[route_survey_column[0]] == row[route_survey_column[0]], 'Total_DIFFERENCE'] = value - 1
#                     break
#         elif next_trans_value:
#             for route in get_valid_routes(row, next_trip_route_code_column):
#                 result_value=process_route(route, counter, 'n')
#                 if result_value:
#                     break
#                 else:
#                     reverse_df.loc[index, 'Type'] = f'{random.choice(["n1","Rev-n1"])}'
#                     route_level_df.loc[route_level_df[route_survey_column[0]] == row[route_survey_column[0]], 'Total_DIFFERENCE'] = value - 1
#                     break
#         else:
#             reverse_df.loc[index, 'Type'] = 'Reverse'
#     else:
#         if prev_trans_value:
#             for route in get_valid_routes(row, prev_trip_route_code_column):
#                 result_value=process_route(route, counter, 'p')
#                 if result_value:
#                     break
#                 else:
#                     reverse_df.loc[index, 'Type'] = f'{random.choice(["p1","Rev-p1"])}'
#                     route_level_df.loc[route_level_df[route_survey_column[0]] == row[route_survey_column[0]], 'Total_DIFFERENCE'] = value - 1
#                     break
#         elif next_trans_value:
#             for route in get_valid_routes(row, next_trip_route_code_column):
#                 result_value=process_route(route, counter, 'n')
#                 if result_value:
#                     break
#                 else:
#                     reverse_df.loc[index, 'Type'] = f'{random.choice(["n1","Rev-n1"])}'
#                     route_level_df.loc[route_level_df[route_survey_column[0]] == row[route_survey_column[0]], 'Total_DIFFERENCE'] = value - 1
#                     break
#         else:
#             reverse_df.loc[index, 'Type'] = 'Reverse'

In [ ]:
# reverse_df[reverse_df['Type']!=''].shape

In [ ]:
all_type_df=copy.deepcopy(reverse_df)

In [ ]:
route_level_df

In [ ]:
route_level_df=copy.deepcopy(new_route_level_df)
route_level_df

In [ ]:

for index, row in reverse_df.iterrows():
    random_value = random.choice([0, 1])
    value = int(route_level_df[route_level_df[route_survey_column[0]] == row[route_survey_column[0]]]['Total_DIFFERENCE'].values)
    
    prev_trans_value = int(df[df['id'] == row['id']][trip_code_column[1]].values)
    next_trans_value = int(df[df['id'] == row['id']][trip_code_column[0]].values)

    counter = [0]

    if value:
        if not random_value:
            reverse_df.loc[row.name, 'Type'] = 'Reverse'
            route_level_df.loc[route_level_df[route_survey_column[0]] == row[route_survey_column[0]], 'Total_DIFFERENCE'] = value - 1
            
        else:
            if prev_trans_value:
                for route in get_valid_routes(row, prev_trip_route_code_column):
                    result_value = process_route(route, counter, 'p')
                    if result_value:
                        break

            elif next_trans_value:
                for route in get_valid_routes(row, next_trip_route_code_column):
                    result_value=process_route(route, counter, 'n')
                    if result_value:
                        break

            else:
                reverse_df.loc[index, 'Type'] = 'Reverse'
                route_level_df.loc[route_level_df[route_survey_column[0]] == row[route_survey_column[0]], 'Total_DIFFERENCE'] = value - 1
        
    else:
        reverse_df.loc[row.name, 'Type'] = ''

In [ ]:
reverse_df['Type'].fillna('',inplace=True)
reverse_df

In [ ]:
reverse_df[reverse_df['Type']!=''].shape